In [19]:
import pandas as pd
import sys
from pathlib import Path

# Locate the repository even if Jupyter was launched from a different folder.
current_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in (current_dir, *current_dir.parents) if (path / "backend").is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not locate the repository root containing backend/")

# repo_root exposes `backend`; backend/ exposes the legacy `config` and
# `scripts` imports still used internally by some modules.
for import_root in (repo_root, repo_root / "backend"):
    import_root = str(import_root)
    if import_root not in sys.path:
        sys.path.insert(0, import_root)

from backend.scripts.data_processing import Data_Processor
from backend.scripts.wallet_size import calculate_total_wallet_size
from backend.scripts.wallet_size_pipeline import *
from backend.scripts.data_aggregation import *
from backend.scripts.calculate_client_score import calculate_client_score


In [2]:
my_processor = Data_Processor()

In [3]:
#company_lvl_df, sens_df = my_processor.extract_external_data_from_pdfs()

In [4]:
#company_lvl_df.to_csv('company_lvl_scraped_new.csv')
#sens_df.to_csv('sens_scraped_new.csv')

In [5]:
company_lvl_df = pd.read_csv('company_lvl_scraped_new.csv')
sens_df = pd.read_csv('sens_scraped_new.csv')

In [6]:
standard_ext_data, standard_sens_data = my_processor.standardize_data(company_lvl_df, sens_df)

In [ ]:
new_company_lvl_df, new_sens_df = my_processor.validate_external_data(standard_ext_data, standard_sens_data)
#TODO Uncomment

In [8]:
new_company_lvl_df, new_sens_df = standard_ext_data, standard_sens_data

In [9]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [10]:
wallet_size, calc_dets, missing_data = calculate_total_wallet_size(company_df=new_company_lvl_df, return_missing_data=True, return_calculation_details=True)

Anglo American: transactional_banking confidence is low because trade_finance: Tier C proxy; missing trade_exposure_value, letters_of_credit_disclosed, imports_value, exports_value, trade_exposure_share
Anglo American: global_markets confidence is low because foreign_exchange: unavailable; missing fx_transaction_value, fx_derivative_notional, fx_exposure_value, foreign_revenue, imports_value, foreign_revenue_share | interest_rates: Tier C proxy; missing interest_rate_derivative_notional, floating_rate_debt | commodities: unavailable; missing commodity_derivative_notional, commodity_exposure_value
Anglo American: investment_banking confidence is low because lending: Tier C proxy; missing bank_loans_and_credit_facilities, bank_loan_debt, explicit_bank_loans, non_bank_debt, identified_bonds, bond_debt | debt_capital_markets: unavailable; missing _event_dcm_value, bond_issue_value, upcoming_bond_maturities_value | equity_capital_markets: unavailable; missing _event_ecm_value, equity_raise_

In [18]:
wallet_size

,transactional_banking,global_markets,investment_banking,total,transactional_banking_confidence,global_markets_confidence,investment_banking_confidence
company,,,,,,,
Anglo American,1.072052e+12,2.564611e+11,3.115581e+11,1.640071e+12,low,low,low
AngloGold Ashanti,4.295637e+11,3.331192e+11,5.793305e+10,8.206159e+11,low,low,low
Aspen Pharmacare,1.850150e+11,3.237400e+10,4.117500e+10,2.585640e+11,low,medium,low
BHP Group,2.079059e+12,2.203206e+12,1.396785e+11,4.421944e+12,low,low,high
Bid Corporation,6.382909e+11,1.796195e+10,2.426123e+10,6.805141e+11,low,low,low
Clicks Group,1.322920e+11,4.042000e+09,5.027000e+09,1.413610e+11,low,low,low
Glencore,1.236143e+13,4.335193e+12,8.023719e+11,1.749900e+13,low,low,low
Gold Fields,3.363716e+11,1.699679e+11,1.131792e+11,6.195187e+11,low,low,low
MTN Group,4.966170e+11,7.045500e+10,1.212810e+11,6.883530e+11,low,low,low


In [11]:
#external_df, sens_df, wallet_size_df = run_wallet_size_pipeline(current_sens_data=sens_df, current_external_data=company_lvl_df, scrape_scope="sens")

In [12]:
#Get Update SENS
opportunities_sens = my_processor.score_sens_opportunities(sens_df)

In [13]:
decayed_sens = my_processor.apply_sens_score_decay(opportunities_sens)

In [14]:
decayed_sens

,Unnamed: 0,company,announcement_date,title,source_document,source_url,event_type,event_value,event_unit,currency,counterparty,target_or_asset,country,expected_completion_date,banking_opportunities,opportunity_summary,extra_notes,transactional_banking_opportunity_score,global_markets_opportunity_score,investment_banking_opportunity_score
0,0,Anglo American,2025-01-29,Anglo American completes sale of minority inte...,Anglo American completes sale of minority inte...,NaN,disposal,1.600000e+09,units,AUD,Zashvin Pty Ltd,33% minority interest in Jellinbah Group Pty Ltd,AU,NaN,"['fx', 'liquidity_management', 'payments']",Inflows from the disposal of Jellinbah stake c...,Completed the sale of a 33.3% minority interes...,0.011125,0.010471,0.003927
1,1,Anglo American,2025-02-17,Anglo American sets out June demerger timeline...,Anglo American sets out June demerger timeline...,NaN,restructuring,1.100000e+09,units,USD,NaN,Anglo American Platinum Limited,ZA,2025-06-30,"['corporate_finance', 'equity_capital_markets'...",Demerger and corporate restructuring of Anglo ...,Sets out timeline for demerger of Anglo Americ...,0.003030,0.006060,0.014393
2,2,Anglo American,2025-02-18,Anglo American agrees sale of nickel business ...,Anglo American agrees sale of nickel business ...,NaN,disposal,5.000000e+08,units,USD,MMG Singapore Resources Pte. Ltd,"Nickel business (Barro Alto and Codemin, plus ...",BR,2025-09-30,"['debt_capital_markets', 'fx', 'payments']",Upfront and contingent cash consideration crea...,Agreed sale of nickel business in Brazil compr...,0.010688,0.011451,0.009161
3,3,Anglo American,2025-02-20,Summarised Preliminary Financial Results for t...,Summarised Preliminary Financial Results for t...,NaN,dividend,8.000000e+08,units,USD,NaN,Final Dividend No. 46,GB,2025-05-07,"['fx', 'liquidity_management', 'payments']",Large-scale dividend payout requires comprehen...,Final dividend of 22 US cents per ordinary sha...,0.013955,0.009303,0.003101
4,4,Anglo American,2025-02-26,ANGLO AMERICAN CAPITAL PLC LAUNCHES CAPPED TEN...,Anglo American Capital plc Launches Capped Ten...,https://clients.dfkingltd.com/angloamerican,bond_issue,4.750000e+08,units,USD,NaN,Various U.S. Dollar and Euro Denominated Notes,GB,NaN,"['debt_capital_markets', 'interest_rates']",Liability management exercise provides opportu...,Launch of capped cash tender offers by subsidi...,0.003248,0.008119,0.013803
5,5,Anglo American,2025-09-04,Results of Anglo American’s accelerated bookbu...,Results of Anglo American’s accelerated bookbu...,NaN,disposal,4.410000e+10,units,ZAR,NaN,Entire remaining c.19.9% interest in Valterra ...,ZA,2025-09-09,"['equity_capital_markets', 'fx', 'liquidity_ma...",Massive capital market placement raising ZAR 4...,Completed the sale of the entire remaining c.1...,0.063138,0.059631,0.063138
6,6,Anglo American,2025-09-09,Anglo American and Teck to combine through a m...,Anglo American and Teck to combine through a m...,NaN,acquisition,4.500000e+09,units,USD,Teck Resources Limited,Teck Resources Limited,CA,2027-03-09,"['corporate_finance', 'credit', 'debt_capital_...",Transformative cross-border merger of equals i...,Agreed merger of equals with Teck Resources Li...,0.058326,0.058326,0.069263
7,7,Anglo American,2026-03-18,RNS PUBLICATION FORM - Issue of Notes,Issue of Notes,NaN,bond_issue,2.300000e+09,units,USD,NaN,"US$2.3 billion Senior Notes due 2031, 2033, an...",US,2026-03-19,"['credit', 'debt_capital_markets', 'interest_r...",Large USD bond issuance providing significant ...,"Priced an issue of US$600M 2031 notes, US$700M...",0.094494,0.236235,0.283482
8,8,Anglo American,2026-05-18,Anglo American agrees sale of steelmaking coal...,Anglo American agrees sale of steelmaking coal...,NaN,disposal,3.875000e+09,units,USD,Dhilmar Limited,Steelmaking Coal Portfolio (Australia),AU,2027-03-31,"['fx', 'liquidity_management', 'payments']",Major divestment transaction generating USD 2....,Agreed to sell its portfolio of steelmaking co...,0.428286,0.377899,0.2015

In [15]:
cross_border_payments = pd.read_csv("../data/cross_border_payments.csv")
trade_finance = pd.read_csv("../data/trade_finance.csv")
transactional_banking = pd.read_csv("../data/transactional_banking.csv")

In [16]:

overlap_df = find_cross_ledger_overlaps(
    cross_border_df=cross_border_payments, 
    transactional_df=transactional_banking,
    day_tolerance=3,
    amount_tolerance_pct=1.0
)
    
# 2. Build the deduplicated client wallet baseline
final_client_table = build_client_wallet_baseline(
    transactional_df=transactional_banking,
    cross_border_df=cross_border_payments,
    trade_finance_df=trade_finance,
    overlap_df=overlap_df
)

In [17]:
final_client_table

,entity_id,entity_name,sector,txn_banking_total_zar,cross_border_total_zar,trade_finance_total_zar,lending_signal_total_zar,lending_signal_txn_count,syn_bank_observed_total_zar
10,E11,Pepkor Holdings,consumer,4.789290e+10,4.880908e+09,4.539648e+09,0.000000e+00,0.0,5.731346e+10
0,E01,BHP Group,mining,3.175649e+10,2.267513e+09,3.949315e+09,0.000000e+00,0.0,3.797331e+10
7,E08,Sanlam,insurance,3.427667e+10,2.314947e+09,5.800268e+08,0.000000e+00,0.0,3.717165e+10
9,E10,Bid Corporation,consumer,1.443446e+10,5.744444e+09,5.461465e+09,2.068692e+08,881.0,2.584723e+10
15,E16,MTN Group,telecoms,1.347148e+10,7.415845e+09,4.391391e+09,2.269990e+08,844.0,2.550572e+10
2,E03,Anglo American,mining,1.795186e+10,1.642429e+09,2.339380e+09,0.000000e+00,0.0,2.193367e+10
8,E09,Shoprite Holdings,consumer,1.336835e+10,3.766578e+09,4.260904e+09,1.753583e+08,793.0,2.157119e+10
1,E02,Glencore,mining,8.304405e+09,3.444080e+09,3.104991e+09,5.500265e+07,83.0,1.490848e+10
17,E18,The Bidvest Group,industrials_pharma,5.777326e+09,2.144064e+09,3.753096e+09,1.012459e+08,409.0,1.177573e+10
18,E19,Aspen Pharmacare,industrials_pharma,4.105327e+09,1.875413e+09,1.292890e+09,5.987651e+07,408.0,7.333507e+09


In [20]:
calculate_client_score(final_client_table, decayed_sens, wallet_size)

,entity_id,entity_name,transactional_banking_total_wallet,transactional_banking_captured_wallet,transactional_banking_wallet_gap,transactional_banking_gap_score,transactional_banking_raw_sens,transactional_banking_sens_score,transactional_banking_current_wallet_share,transactional_banking_relationship_score,transactional_banking_score,global_markets_total_wallet,global_markets_captured_wallet,global_markets_wallet_gap,global_markets_gap_score,global_markets_raw_sens,global_markets_sens_score,global_markets_current_wallet_share,global_markets_relationship_score,global_markets_score,investment_banking_total_wallet,investment_banking_captured_wallet,investment_banking_wallet_gap,investment_banking_gap_score,investment_banking_raw_sens,investment_banking_sens_score,investment_banking_current_wallet_share,investment_banking_relationship_score,investment_banking_score,total_score
0,E11,Pepkor Holdings,2.290290e+11,5.243255e+10,1.765964e+11,0.008474,0.760894,0.075568,0.228934,0.95,0.129464,1.690000e+10,4.880908e+09,1.201909e+10,0.001486,0.462050,0.068643,0.288811,0.85,0.113200,1.546600e+10,0.000000e+00,1.546600e+10,0.006221,1.175718,0.107773,0.000000,0.125,0.058720,0.123145
1,E01,BHP Group,2.079059e+12,3.570580e+10,2.043353e+12,0.098051,0.333623,0.033134,0.017174,0.60,0.122279,2.203206e+12,2.267513e+09,2.200938e+12,0.272100,0.220256,0.032722,0.001029,0.10,0.159139,1.396785e+11,0.000000e+00,1.396785e+11,0.056185,0.485822,0.044533,0.000000,0.125,0.058406,0.138749
2,E08,Sanlam,1.265650e+11,3.485670e+10,9.170830e+10,0.004401,0.520673,0.051711,0.275406,1.00,0.122885,1.857800e+10,2.314947e+09,1.626305e+10,0.002011,0.399976,0.059422,0.124607,0.75,0.099774,1.940900e+10,0.000000e+00,1.940900e+10,0.007807,0.489744,0.044893,0.000000,0.125,0.034361,0.106446
3,E10,Bid Corporation,6.382909e+11,1.989592e+10,6.183950e+11,0.029674,0.920480,0.091417,0.031171,0.85,0.136404,1.796195e+10,5.744444e+09,1.221751e+10,0.001510,0.244568,0.036334,0.319812,0.90,0.105289,2.426123e+10,2.068692e+08,2.405436e+10,0.009676,0.784245,0.071888,0.008527,0.950,0.128593,0.135536
4,E16,MTN Group,4.966170e+11,1.786287e+10,4.787541e+11,0.022973,0.616772,0.061255,0.035969,0.90,0.125988,7.045500e+10,7.415845e+09,6.303916e+10,0.007793,0.291876,0.043362,0.105256,0.70,0.091242,1.212810e+11,2.269990e+08,1.210540e+11,0.048693,0.544355,0.049899,0.001872,0.750,0.119306,0.121463
5,E03,Anglo American,1.072052e+12,2.029124e+10,1.051760e+12,0.050469,0.686290,0.068159,0.018927,0.65,0.117498,2.564611e+11,1.642429e+09,2.548187e+11,0.031503,0.777496,0.115507,0.006404,0.20,0.081954,3.115581e+11,0.000000e+00,3.115581e+11,0.125322,0.661814,0.060665,0.000000,0.125,0.099427,0.108421
6,E09,Shoprite Holdings,6.788900e+11,1.762925e+10,6.612607e+11,0.031731,0.408573,0.040577,0.025968,0.70,0.102096,3.003700e+10,3.766578e+09,2.627042e+10,0.003248,0.168777,0.025074,0.125398,0.80,0.091653,1.431700e+10,1.753583e+08,1.414164e+10,0.005688,0.160660,0.014727,0.012248,1.000,0.108735,0.101839
7,E02,Glencore,1.236143e+13,1.140940e+10,1.235002e+13,0.592620,0.000000,0.000000,0.000923,0.05,0.301310,4.335193e+12,3.444080e+09,4.331749e+12,0.535531,0.000000,0.000000,0.000794,0.05,0.272765,8.023719e+11,5.500265e+07,8.023169e+11,0.322726,0.000000,0.000000,0.000069,0.300,0.191363,0.289193
8,E18,The Bidvest Group,3.624704e+11,9.530423e+09,3.529400e+11,0.016936,0.182659,0.018141,0.026293,0.75,0.090724,6.603461e+10,2.144064e+09,6.389055e+10,0.007899,0.138065,0.020511,0.032469,0.40,0.052154,4.236670e+10,1.012459e+08,4.226545e+10,0.017001,0.362680,0.033245,0.002390,0.800,0.101799,0.086376
9,E19,Aspen Pharmacare,1.850150e+11,5.398218e+09,1.796168e+11,0.008619,0.085707,0.008512,0.029177,0.80,0.087714,3.237400e+10,1.875413e+09,3.049859e+10,0.003771,0.128560,0.019099,0.057930,0.55,0.064525,4.117500e+10,5.987651e+07,4.111512e+10,0.016538,0.145702,0.013356,0.001454,0.700,0.083611,0.084228
